# Chapter 9 — Reducing the number of dimensions · Annotated solution

*Maths for AI, Step by Step*, Belk Siri — companion notebook.

[Open in Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/en/ch09_corrige.ipynb) · Companion page: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/en.html

Each answer first presents the calculation by hand, as in the book's solutions, then checks it in Python.

**Convention.** As in the book, means, variances and covariances divide by $n$.

**Contents.** Data · The computations of lessons 9.1 to 9.3 · Exercises 9.1 to 9.6

**Before opening Jupyter:** the exercises in this chapter are done first on paper, with a calculator. The cells below are there to **check** your results.

**Color code:** blue = homes; black diamond = G; orange = projections; dashed magenta = what the projection loses.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# The book's color code: homes in blue, G as a black diamond, projections in orange,
# what the projection loses in magenta (dashed)
BLUE, ORANGE, MAGENTA, INK = '#2a78d6', '#eb6834', '#c2185b', '#0b0b0b'


def project(X, u):
    # coordinates of the points (rows of X) on the direction u, scaled to norm 1 (equation 9.1)
    u = np.asarray(u, dtype=float)
    return np.asarray(X, dtype=float) @ (u / np.linalg.norm(u))


def cov_matrix(X):
    # covariance matrix of the columns of X, with a division by n (equation 9.2)
    X = np.asarray(X, dtype=float)
    D = X - X.mean(axis=0)
    return D.T @ D / len(X)

## Data

The identifier D2.1 is that of the book and of the CSV file. The small datasets of the exercises are copied from the exercise statements.

In [ ]:
# Chapter 9 data, copied from the book's CSV files (do not modify)

# D2.1 — the seven homes (A to F: reference sales; G: the home to estimate)
# CSV columns → notebook names: logement → home, surface_m2 → floor_area (m²),
# distance_km → distance (km, distance to the center), prix_kEUR → price (k€)
homes = pd.DataFrame([
    ('A', 32, 3.0, 118),
    ('B', 45, 1.5, 172),
    ('C', 54, 2.5, 185),
    ('D', 70, 1.0, 245),
    ('E', 85, 2.5, 279),
    ('F', 98, 0.5, 335),
    ('G', 62, 2.5, 209)
], columns=['home', 'floor_area', 'distance', 'price'])

reference = homes[homes['home'] != 'G']          # A to F: reference means and standard deviations
mu = reference[['floor_area', 'distance']].mean().to_numpy()
sigma = reference[['floor_area', 'distance']].std(ddof=0).to_numpy()   # division by n
Z = (reference[['floor_area', 'distance']].to_numpy() - mu) / sigma   # standardized scores of A to F
z_G = (homes.loc[homes['home'] == 'G', ['floor_area', 'distance']].to_numpy()[0] - mu) / sigma
names = list(reference['home'])
print(homes.to_string(index=False))

### Data variables

Code names are in English. For the columns, the last column gives the name used in the book's CSV files.

| Name | Meaning | Unit | CSV column |
|:--|:--|:--|:--|
| `home` | home identifier (A, B…) | — | `logement` |
| `floor_area` | living area of the home | m² | `surface_m2` |
| `price` | sale price | k€ | `prix_kEUR` |
| `distance` | distance to the city center | km | `distance_km` |
| `trainee` | trainee identifier | — | — |
| `math_score` | math score | points out of 20 | — |
| `writing_score` | writing score | points out of 20 | — |

## The computations of the lessons, in Python

These cells redo the computations of lessons 9.1 to 9.3.

In [ ]:
# Lesson 9.1 — projecting onto an axis (equation 9.1): the floor-area axis, then the diagonal u = (1, −1)/√2
u_area = np.array([1.0, 0.0])
u_diag = np.array([1.0, -1.0]) / np.sqrt(2)
t = project(Z, u_diag)
t_G = project(z_G, u_diag)
print(pd.DataFrame({'home': names, 'z_area': Z[:, 0], 'z_distance': Z[:, 1], 't': t}).round(3).to_string(index=False))
print(f"G: t = {t_G:.3f}")
print(f"variance kept: floor-area axis {np.mean(project(Z, u_area) ** 2):.3f}, diagonal {np.mean(t ** 2):.3f}")
print(f"total variance {np.sum(Z.var(axis=0)):.3f} = kept {np.mean(t ** 2):.3f} + lost {np.sum(Z.var(axis=0)) - np.mean(t ** 2):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, u, title in zip(axes, (u_area, u_diag), ('floor-area axis', 'diagonal')):
    shadows = np.outer(project(Z, u), u)
    for p, s in zip(Z, shadows):
        ax.plot([p[0], s[0]], [p[1], s[1]], color=MAGENTA, linestyle='--', linewidth=1)
    ax.axline((0, 0), u, color=ORANGE, linewidth=1)
    ax.scatter(Z[:, 0], Z[:, 1], color=BLUE, zorder=3)
    ax.scatter(shadows[:, 0], shadows[:, 1], color=ORANGE, zorder=3)
    ax.scatter([z_G[0]], [z_G[1]], marker='D', color=INK, zorder=3)
    for n, p in zip(names + ['G'], list(Z) + [z_G]):
        ax.annotate(n, p, xytext=(4, 4), textcoords='offset points')
    ax.set_aspect('equal'); ax.set_title(title); ax.set_xlabel('z floor area'); ax.set_ylabel('z distance')
plt.tight_layout(); plt.show()

In [ ]:
# Lesson 9.2 — the covariance matrix (equation 9.2) and its eigendirections
products = Z[:, 0] * Z[:, 1]
print('products z_s × z_d:', products.round(3), '→ sum', round(products.sum(), 3))
C = cov_matrix(Z)
print('covariance matrix C =\n', C.round(3))
for vec in ([1, 0], [1, -1], [1, 1]):
    print(f"C × {vec} = {(C @ np.array(vec, dtype=float)).round(3)}")
u1, u2 = u_diag
print(f"variance on the diagonal, from C alone: {C[0, 0] * u1 ** 2 + 2 * C[0, 1] * u1 * u2 + C[1, 1] * u2 ** 2:.3f}")

In [ ]:
# Lesson 9.3 — PCA: eigenvalues sorted in decreasing order, share of variance kept (equation 9.3)
eigenvalues, eigenvectors = np.linalg.eigh(C)          # eigh: symmetric matrix, increasing order
order = np.argsort(eigenvalues)[::-1]
eigenvalues, eigenvectors = eigenvalues[order], eigenvectors[:, order]
v1 = eigenvectors[:, 0] * np.sign(eigenvectors[0, 0])  # a direction and its opposite describe the same axis
print('eigenvalues:', eigenvalues.round(3), '→ sum', round(eigenvalues.sum(), 3))
print('first eigendirection:', v1.round(3))
print('shares kept (%):', (100 * eigenvalues / eigenvalues.sum()).round(1))
pc1 = project(Z, v1)
print('sales sorted by first component:', [names[i] for i in np.argsort(pc1)],
      '; prices (k€):', list(reference['price'].to_numpy()[np.argsort(pc1)]))

# B and E: close on the component, far apart in the plane
q = project(Z, [1, 1])
iB, iE = names.index('B'), names.index('E')
print(f"B and E: gap on the component {abs(pc1[iE] - pc1[iB]):.2f}, distance in the plane "
      f"{np.linalg.norm(Z[iE] - Z[iB]):.2f}, gap on the second component {abs(q[iE] - q[iB]):.2f}")

# Going further: the SVD of Z gives the same eigenvalues, s_k² / n
s = np.linalg.svd(Z, compute_uv=False)
print(f"singular values: {s.round(3)}; s² / n = {(s ** 2 / len(Z)).round(3)}")

## Exercise 9.1 — Projecting onto two axes

*Concepts: projection and variance kept (lesson 9.1, equation 9.1).* Three points, $P_1 = (4, 2)$, $P_2 = (1, 3)$ and $P_3 = (-2, 1)$, and two directions, $\mathbf{u} = (0.8, 0.6)$ and $\mathbf{v} = (-0.6, 0.8)$.

### a) to d) The calculation by hand

a) $0.8^2 + 0.6^2 = 0.64 + 0.36 = 1$ and $0.6^2 + 0.8^2 = 1$. Moreover, $\mathbf{u} \cdot \mathbf{v} = 0.8 \times (-0.6) + 0.6 \times 0.8 = 0$.

b) On $\mathbf{u}$: $P_1$: $3.2 + 1.2 = 4.4$; $P_2$: $0.8 + 1.8 = 2.6$; $P_3$: $-1.6 + 0.6 = -1.0$. On $\mathbf{v}$: $P_1$: $-2.4 + 1.6 = -0.8$; $P_2$: $-0.6 + 2.4 = 1.8$; $P_3$: $1.2 + 0.8 = 2.0$.

c) $4.4^2 + (-0.8)^2 = 19.36 + 0.64 = 20$, and $\lVert P_1 \rVert^2 = 16 + 4 = 20$.

d) On $\mathbf{u}$: mean 2.0, deviations 2.4, 0.6 and −3.0, variance $(5.76 + 0.36 + 9)/3 = 5.04$. On $\mathbf{v}$: mean 1.0, deviations −1.8, 0.8 and 1.0, variance $(3.24 + 0.64 + 1)/3 \approx 1.63$. The axis $\mathbf{u}$ keeps the most variance. The variances of the original coordinates are 6 and $2/3 \approx 0.67$: the total variance is 6.67, and $5.04 + 1.63 = 6.67$. Nothing is lost when you keep two perpendicular directions.

The code below checks these results.

In [ ]:
# The three points (one row per point) and the two directions
points = np.array([[4, 2], [1, 3], [-2, 1]], dtype=float)
u = np.array([0.8, 0.6])
v = np.array([-0.6, 0.8])

In [ ]:
# b) and d): coordinates and variances
coords_u = points @ u
coords_v = points @ v
var_u = np.var(coords_u)
var_v = np.var(coords_v)

In [ ]:
# Check of a) to d)
if coords_u is None or coords_v is None or var_u is None or var_v is None:
    print('not yet completed. Complete the four lines.')
else:
    print(f"a) norms {np.linalg.norm(u):.3f} and {np.linalg.norm(v):.3f}, dot product {u @ v:.3f}")
    print('b) on u:', np.round(coords_u, 2), '; on v:', np.round(coords_v, 2), '→',
          'correct' if np.allclose(coords_u, [4.4, 2.6, -1.0]) and np.allclose(coords_v, [-0.8, 1.8, 2.0]) else 'check again')
    print(f"c) P1: {coords_u[0] ** 2 + coords_v[0] ** 2:.2f} versus a squared norm of {points[0] @ points[0]:.2f}")
    total = np.sum(points.var(axis=0))
    print(f"d) variances {var_u:.2f} and {var_v:.2f}, sum {var_u + var_v:.2f}, total variance {total:.2f} →",
          'correct' if abs(var_u - 5.04) < 1e-9 and abs(var_v - 1.6267) < 1e-3 and abs(var_u + var_v - total) < 1e-9 else 'check again')

## Exercise 9.2 — A covariance matrix by hand

*Concepts: covariance (lesson 1.5), covariance matrix (lesson 9.2, equation 9.2).* Four trainees took two tests, each graded out of 20 (made-up dataset).

- `trainee` — trainee, S1 to S4;
- `math_score` — score on the math test, out of 20, in points;
- `writing_score` — score on the writing test, out of 20, in points.

In [ ]:
# The four trainees (exercises 9.2 to 9.5)
trainees = pd.DataFrame({
    'trainee':       ['S1', 'S2', 'S3', 'S4'],
    'math_score':    [15, 13, 9, 11],
    'writing_score': [11, 13, 9, 7],
})
scores = trainees[['math_score', 'writing_score']].to_numpy(dtype=float)
print(trainees.to_string(index=False))

### a) to c) The calculation by hand

a) Means: `math_score` $48/4 = 12$, `writing_score` $40/4 = 10$. Deviations: S1 $(3, 1)$, S2 $(1, 3)$, S3 $(-3, -1)$, S4 $(-1, -3)$.

b) Math: $(9 + 1 + 9 + 1)/4 = 5$. Writing: $(1 + 9 + 1 + 9)/4 = 5$. Covariance: $(3 + 3 + 3 + 3)/4 = 3$.

c) $M = \begin{pmatrix} 5 & 3 \\ 3 & 5 \end{pmatrix}$ and $r = 3/\sqrt{5 \times 5} = 0.6$.

The code below checks these results.

In [ ]:
# a) to c): deviations, covariance matrix, correlation
deviations = scores - scores.mean(axis=0)
M = deviations.T @ deviations / 4
r = M[0, 1] / np.sqrt(M[0, 0] * M[1, 1])

In [ ]:
# Check of a) to c)
if deviations is None or M is None or r is None:
    print('not yet completed. Complete the three lines.')
else:
    print('a) means:', scores.mean(axis=0), '; deviations:', deviations.tolist())
    print('b) and c) M =', np.asarray(M).tolist(), f"; r = {r:.2f} →",
          'correct' if np.allclose(M, [[5, 3], [3, 5]]) and abs(r - 0.6) < 1e-9 else 'check again')
    print('check with cov_matrix:', cov_matrix(scores).tolist())

## Exercise 9.3 — Finding the eigendirections

*Concepts: eigendirection and eigenvalue (lesson 9.2).* We go back to the matrix of exercise 9.2, $M = \begin{pmatrix} 5 & 3 \\ 3 & 5 \end{pmatrix}$.

### a) to d) The calculation by hand

a) $M (1, 0) = (5, 3)$: the vector has turned, so it is not an eigendirection.

b) $M (1, 1) = (8, 8) = 8\,(1, 1)$ and $M (1, -1) = (2, -2) = 2\,(1, -1)$. The eigenvalues are 8 and 2.

c) $8 + 2 = 10 = 5 + 5$. The best direction, $(1, 1)$, keeps $8/10 = 80$%. The math axis alone keeps its variance, 5, that is, 50%.

d) $\begin{pmatrix} 1 & r \\ r & 1 \end{pmatrix} (1, 1) = (1 + r, r + 1) = (1 + r)(1, 1)$ and $\begin{pmatrix} 1 & r \\ r & 1 \end{pmatrix} (1, -1) = (1 - r, r - 1) = (1 - r)(1, -1)$. With $r = -0.603$: $1 + r = 0.397$ for $(1, 1)$ and $1 - r = 1.603$ for $(1, -1)$, as in lesson 9.2.

The code below checks these results with `np.linalg.eigh`.

In [ ]:
# b) and c): eigenvalues and share kept
M93 = np.array([[5.0, 3.0], [3.0, 5.0]])
eigenvalues_93 = np.sort(np.linalg.eigh(M93)[0])[::-1]
best_share = eigenvalues_93[0] / eigenvalues_93.sum()

In [ ]:
# Check of a) to d)
if eigenvalues_93 is None or best_share is None:
    print('not yet completed. Complete the two lines.')
else:
    for vec in ([1, 0], [1, 1], [1, -1]):
        print(f"a) and b) M × {vec} = {(M93 @ np.array(vec, dtype=float)).tolist()}")
    print(f"eigenvalues {eigenvalues_93.round(3).tolist()}, share of the best direction {100 * best_share:.0f}% →",
          'correct' if np.allclose(eigenvalues_93, [8, 2]) and abs(best_share - 0.8) < 1e-9 else 'check again')
    r_houses = -0.603
    print('d) r = −0.603: eigenvalues', np.sort(np.linalg.eigh(np.array([[1, r_houses], [r_houses, 1]]))[0])[::-1].round(3).tolist())

## Exercise 9.4 — Computing a first component

*Concepts: PCA (lesson 9.3), projection (equation 9.1).* We go back to the deviations from the mean of exercise 9.2. Here, we do not standardize: the two tests have the same unit and the same variance. The first eigendirection is $\mathbf{v}_1 = (1, 1)/\sqrt{2}$.

### a) to d) The calculation by hand

a) S1: $0.707 \times (3 + 1) \approx 2.83$; S2: $0.707 \times (1 + 3) \approx 2.83$; S3 and S4: $-2.83$. Exactly, $4/\sqrt{2} = 2\sqrt{2}$, whose square is 8. The variance is $(8 + 8 + 8 + 8)/4 = 8$, the first eigenvalue.

b) $8/10 = 80$%.

c) $t_1\, \mathbf{v}_1 = 2\sqrt{2} \times (1, 1)/\sqrt{2} = (2, 2)$. Reconstructed scores: $12 + 2 = 14$ in math and $10 + 2 = 12$ in writing, instead of 15 and 11. The error is $(3, 1) - (2, 2) = (1, -1)$, of length $\sqrt{2} \approx 1.41$. It is the same for all four trainees, and the mean of the squares, 2, is the second eigenvalue: the variance lost.

d) The first coordinate is proportional to the sum of the two deviations: it measures the overall level. The second is proportional to their difference: it tells whether a trainee does better in math or in writing.

The code below checks a) to c).

In [ ]:
# a) and c): coordinates, then reconstructed scores of S1
v1_trainees = np.array([1.0, 1.0]) / np.sqrt(2)
t_trainees = (scores - scores.mean(axis=0)) @ v1_trainees
reconstructed_S1 = scores.mean(axis=0) + t_trainees[0] * v1_trainees

In [ ]:
# Check of a) to c)
if t_trainees is None or reconstructed_S1 is None:
    print('not yet completed. Complete the two lines.')
else:
    var_t = np.mean(t_trainees ** 2)
    error = scores[0] - reconstructed_S1
    print('a) coordinates:', t_trainees.round(2).tolist(), f"; variance {var_t:.2f}")
    print(f"b) share kept {100 * var_t / np.sum(scores.var(axis=0)):.0f}%")
    print('c) reconstructed scores of S1:', reconstructed_S1.round(2).tolist(), '; error:', error.round(2).tolist(),
          f"; length {np.linalg.norm(error):.2f} →",
          'correct' if np.allclose(t_trainees, [2 * np.sqrt(2)] * 2 + [-2 * np.sqrt(2)] * 2)
          and np.allclose(reconstructed_S1, [14, 12]) else 'check again')

## Exercise 9.5 — Close after projection?

*Concepts: limits of projection (lesson 9.3, equation 9.3).* We go back to the coordinates of exercise 9.4.

### a) to d) The calculation by hand

a) Same coordinates, $2\sqrt{2} \approx 2.83$: S1 and S2 coincide on the first component. In the plane, their difference is $(15 - 13, 11 - 13) = (2, -2)$, and their distance $\sqrt{8} \approx 2.83$.

b) The difference $\mathbf{x}_i - \mathbf{x}_j$ splits into $(t_i - t_j)\, \mathbf{v}_1 + (q_i - q_j)\, \mathbf{v}_2$, where $q$ is the coordinate on $\mathbf{v}_2$. The two pieces are perpendicular, so $d^2 = (t_i - t_j)^2 + (q_i - q_j)^2 \ge (t_i - t_j)^2$.

c) $\lvert t_1 - t_3 \rvert = 4\sqrt{2} \approx 5.66$: their distance is at least 5.66. Indeed, $(15 - 9, 11 - 9) = (6, 2)$, and $\sqrt{40} \approx 6.32$.

d) Gap on the first component: $0.126 - (-0.327) \approx 0.45$. Distance in the plane: 2.08. The part lost is their gap on the second component, about 2.03, since $0.45^2 + 2.03^2 \approx 4.32 \approx 2.08^2$.

The code below checks a), c) and d).

In [ ]:
# a) and c): distances in the plane of the two scores
distance_S1_S2 = np.linalg.norm(scores[0] - scores[1])
distance_S1_S3 = np.linalg.norm(scores[0] - scores[2])

In [ ]:
# Check of a), c) and d)
if distance_S1_S2 is None or distance_S1_S3 is None:
    print('not yet completed. Complete the two lines.')
else:
    t9 = (scores - scores.mean(axis=0)) @ (np.array([1.0, 1.0]) / np.sqrt(2))
    print(f"a) S1 and S2: projected gap {abs(t9[0] - t9[1]):.2f}, distance {distance_S1_S2:.2f}")
    print(f"c) S1 and S3: projected gap {abs(t9[0] - t9[2]):.2f} ≤ distance {distance_S1_S3:.2f} →",
          'correct' if abs(distance_S1_S2 - np.sqrt(8)) < 1e-9 and abs(distance_S1_S3 - np.sqrt(40)) < 1e-9 else 'check again')
    iB, iE = names.index('B'), names.index('E')
    t_BE = project(Z[[iB, iE]], [1, -1])
    q_BE = project(Z[[iB, iE]], [1, 1])
    print(f"d) B and E: gap on the component {abs(t_BE[1] - t_BE[0]):.2f}, distance {np.linalg.norm(Z[iE] - Z[iB]):.2f}, "
          f"part lost {abs(q_BE[1] - q_BE[0]):.2f}")

## Exercise 9.6 — Challenge: PCA without standardizing

*Concepts: covariance matrix (equation 9.2), PCA and units (lesson 9.3).* For sales A to F, we work on the raw deviations from the mean, without standardizing.

- `floor_area` — floor area of the home, in m² (dataset D2.1, column `surface_m2`);
- `distance` — distance to the center, in km (dataset D2.1, column `distance_km`).

### a) to d) The calculation by hand

a) The deviations of `floor_area` are −32, −19, −10, 6, 21 and 34 m²; those of `distance`, 7, −2, 4, −5, 4 and −8 sixths of a km. Multiplying them pairwise: $-224 + 38 - 40 - 30 + 84 - 272 = -444$ m² × sixths of a km, that is, $-444/6 = -74$ m²·km. The covariance is $-74/6 \approx -12.33$ m²·km.

b) $M = \begin{pmatrix} 519.67 & -12.33 \\ -12.33 & 0.806 \end{pmatrix}$. Then $M \mathbf{w} \approx (519.96, -12.35)$ for $\mathbf{w} = (1, -0.0238)$. The ratio of the two coordinates is $-12.35/519.96 \approx -0.0238$, as for $\mathbf{w}$: the direction has not turned. Its eigenvalue is about 519.96.

c) The total variance is $519.67 + 0.806 \approx 520.47$. The share kept is $519.96/520.47 \approx 99.9$%. The direction is almost the floor-area axis: the first component is nothing but floor area.

d) Standardized, PCA kept 80.1% on a direction that mixed floor area and distance equally: the raw “99.9%” only measures the gap between the units. In meters, the variance of the distance would become 805,556 m², and the first component would be almost the distance axis (99.96%). Variables with different units must be standardized.

The code below checks these results.

In [ ]:
# a) and b): raw covariance matrix, then M × w
raw = reference[['floor_area', 'distance']].to_numpy(dtype=float)
M_raw = cov_matrix(raw)
w = np.array([1.0, -0.0238])
Mw = M_raw @ w

In [ ]:
# Check of a) to d)
if M_raw is None or Mw is None:
    print('not yet completed. Complete the two lines.')
else:
    share = np.linalg.eigvalsh(M_raw)[-1] / np.trace(M_raw)
    raw_m = raw * [1, 1000]                       # distance in meters
    share_m = np.linalg.eigvalsh(cov_matrix(raw_m))[-1] / np.trace(cov_matrix(raw_m))
    print('a) and b) M =', M_raw.round(3).tolist(), '; M × w =', Mw.round(2).tolist(), f"; ratio {Mw[1] / Mw[0]:.4f}")
    print(f"c) share kept {100 * share:.1f}% (total variance {np.trace(M_raw):.2f}) →",
          'correct' if abs(M_raw[0, 1] + 74 / 6) < 1e-9 and abs(Mw[1] / Mw[0] + 0.0238) < 5e-5 else 'check again')
    print(f"d) distance in meters: variance {cov_matrix(raw_m)[1, 1]:.0f} m², share kept {100 * share_m:.2f}%")